# Üye notebook'u – `ilkin`
**Yöntem ailesi:** Morfoloji + distance transform + watershed ile birbirine değen hücrelerin ayrılması

Bu notebook **yalnızca sekans 01** (geliştirme + doğrulama) üzerinde çalışır; test referanslarına dokunmaz.
Çıktılar `seeds/<seed>/` klasörüne yazılır.

1. Kurulum, veri indirme, seçimin doğrulanması
2. Geliştirme/doğrulama referans sayıları
3. Kontrollü deneyler (geliştirme kümesi) → `experiments_ilkin.csv`
4. En iyi 3 aday → doğrulama → üyenin yöntemi `method_ilkin.json`

> 🟢 Bitince commit: `seeds/<seed>/*_ilkin.*` + bu notebook (çıktıları temizlenmiş olarak). Test **kilidi** ortak `cross_seed.ipynb` içinde yapılır.

In [ ]:
# --- Kurulum: Colab'da repoyu klonlar; yerelde notebooks/ klasöründen çalıştırılabilir ---
import os, sys, subprocess
REPO_URL = "https://github.com/James007Bond-code/goruntu-isleme-hucre-sayimi-grup-odevi.git"
REPO_NAME = "goruntu-isleme-hucre-sayimi-grup-odevi"
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if not os.path.isdir("src"):
    if not os.path.isdir(REPO_NAME):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir(REPO_NAME)
sys.path.insert(0, os.getcwd())
print("Çalışma klasörü:", os.getcwd())

In [ ]:
MEMBER = "ilkin"

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src.data import download_dataset, load_image, read_gt_count
from src.selection import build_selection, load_selection
from src.pipeline import BASELINE, E11_SULEYMAN, make, param_str
from src.evaluation import (evaluate, warmup, run_experiments, rank_candidates,
                            evaluate_method, save_json)

os.makedirs(C.FIG_DIR, exist_ok=True); os.makedirs(C.FIG_IMG_DIR, exist_ok=True)
plt.rcParams["figure.dpi"] = 110
n01, n02 = download_dataset()
print(f"Sekans 01: {n01} kare | Sekans 02: {n02} kare")

SEED = C.SEEDS[MEMBER]
print(f"Üye: {C.MEMBERS[MEMBER]['name']} | seed = {C.MEMBERS[MEMBER]['student_id'][-5:]} + 2026 = {SEED}")

# Seçim ilk commit'te kaydedildi; burada yeniden üretilip birebir aynı olduğu doğrulanır
sel = load_selection(SEED)
regen, _ = build_selection(SEED, n01, n02)
assert sel.equals(regen.astype(sel.dtypes.to_dict())), "Kayıtlı seçim veri setiyle uyuşmuyor!"
print(sel.groupby("group")["frame_idx"].apply(list).to_string())

## Geliştirme ve doğrulama referansları (yalnızca sekans 01)

In [ ]:
devval = sel[sel.group.isin(["Development", "Validation"])].copy()
devval["gt_count"] = [read_gt_count(r.sequence, r.frame_idx) for r in devval.itertuples()]
assert devval["gt_count"].notna().all(), "Eksik referans var"
devval.to_csv(os.path.join(C.seed_dir(SEED), "dev_val_references.csv"), index=False)
dev_df = devval[devval.group == "Development"].reset_index(drop=True)
val_df = devval[devval.group == "Validation"].reset_index(drop=True)

HM_REF = dev_df.loc[0, "filename"]          # histogram matching referansı: YALNIZCA kendi geliştirme kümesinden
hm_ref = load_image("01", HM_REF)
warmup(hm_ref)
print(devval[["group", "frame_idx", "gt_count"]].to_string(index=False))

## Kontrollü deneyler (geliştirme kümesi)
Her deney referans deneyden **tek bir faktörle** ayrılır (`Referans`, `Degisen`).

In [ ]:
E_REF = make(BASELINE, intensity="clahe", clahe_clip=3.0, clahe_grid=[8, 8], filter="gaussian", filter_k=5,
             morph="open_close", morph_k=5, min_area=150)
EXPERIMENTS = [
    ("W01", "Baseline: orijinal + Otsu + CC", "-", "-", BASELINE),
    ("W02", "CLAHE + Gauss5 + Otsu + Open→Close 5×5 + CC", "W01", "pipeline", E_REF),
    ("W03", "W02 + watershed (min_dist=16)", "W02", "ayırma", make(E_REF, separation="watershed", ws_min_dist=16)),
    ("W04", "Watershed min_dist=10", "W03", "ws_min_dist", make(E_REF, separation="watershed", ws_min_dist=10)),
    ("W05", "Watershed min_dist=22", "W03", "ws_min_dist", make(E_REF, separation="watershed", ws_min_dist=22)),
    ("W06", "Close→Open (sıra) + watershed", "W03", "morfoloji sırası", make(E_REF, morph="close_open", separation="watershed", ws_min_dist=16)),
    ("W07", "SE 3×3 + watershed", "W03", "SE boyutu", make(E_REF, morph_k=3, separation="watershed", ws_min_dist=16)),
    ("W08", "Erosion 5×5 + CC (ayırma için aşındırma)", "W02", "morfoloji", make(E_REF, morph="erosion")),
    ("W09", "Adaptif(71,-12) + watershed", "W03", "segmentasyon", make(E_REF, segmentation="adaptive", adapt_block=71, adapt_c=-12, morph="none", min_area=100, separation="watershed", ws_min_dist=16)),
    # TODO (İlkin): dilation, SE 7×7 ve watershed öncesi erosion gibi deneylerle genişlet
]

df_exp, df_pf = run_experiments(EXPERIMENTS, dev_df, hm_ref, member=MEMBER, seed=SEED)
df_exp.to_csv(os.path.join(C.seed_dir(SEED), f"experiments_{MEMBER}.csv"), index=False)
df_pf.to_csv(os.path.join(C.seed_dir(SEED), f"experiments_per_frame_{MEMBER}.csv"), index=False)
print(df_exp[["Deney_ID", "Aciklama", "Degisen", "Dev_MAE", "Dev_RMSE", "Dev_Bias", "Sure_ms"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(df_exp["Deney_ID"], df_exp["Dev_MAE"].clip(upper=20), color="#1E88E5")
for i, v in enumerate(df_exp["Dev_MAE"]):
    ax.text(i, min(v, 20) + 0.3, f"{v:.1f}", ha="center", fontsize=8)
ax.set_ylabel("Geliştirme MAE (20'de kırpıldı)"); ax.grid(axis="y", alpha=0.3)
ax.set_title(f"{C.MEMBERS[MEMBER]['name']} – seed {SEED} – kontrollü deneyler", fontsize=10)
plt.tight_layout(); plt.savefig(f"{C.FIG_DIR}/experiments_{MEMBER}_{SEED}.png", dpi=200); plt.show()

## Doğrulama ve üyenin yöntemi
Baseline dışındaki en düşük geliştirme MAE'li 3 deney aday olur. Seçim **doğrulama** MAE'sine göredir (eşitlikte RMSE, karmaşıklık, süre).

In [ ]:
cfg_by_id = {eid: cfg for eid, *_, cfg in EXPERIMENTS}
top3 = df_exp[df_exp["Deney_ID"] != EXPERIMENTS[0][0]].sort_values(["Dev_MAE", "Dev_RMSE", "Deney_ID"]).head(3)
CANDIDATES = {f"{r.Deney_ID}: {r.Aciklama}": cfg_by_id[r.Deney_ID] for r in top3.itertuples()}
df_val = rank_candidates(CANDIDATES, val_df, hm_ref)
df_val.insert(0, "Uye", MEMBER); df_val.insert(1, "Seed", SEED)
df_val.to_csv(os.path.join(C.seed_dir(SEED), f"validation_{MEMBER}.csv"), index=False)
print(df_val.to_string(index=False))

best_name = df_val.loc[0, "Pipeline"]
best_cfg = CANDIDATES[best_name]
method = {"name": f"M_{MEMBER}", "owner": MEMBER, "seed": SEED, "experiment": best_name,
          "config": best_cfg, "hm_ref": HM_REF if best_cfg.get("intensity") == "hist_match" else None,
          "validation_own_seed": {k: float(df_val.loc[0, k]) for k in ["Validation_MAE", "Validation_RMSE", "Ortalama_Sure_ms"]}}
save_json(method, os.path.join(C.seed_dir(SEED), f"method_{MEMBER}.json"))
print("\nÜyenin yöntemi:", best_name, "|", param_str(best_cfg))